<a href="https://colab.research.google.com/github/wajd-mq/wajd-mq/blob/main/PDF_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 PDF RAG Chatbot — End-to-End Project

A complete Retrieval-Augmented Generation (RAG) chatbot that:

1. Accepts an uploaded PDF
2. Extracts and cleans its text
3. Splits it into chunks
4. Embeds the chunks with Sentence-Transformers
5. Stores embeddings in ChromaDB
6. Retrieves relevant chunks for a user question
7. Generates a grounded answer with FLAN-T5
8. Wraps everything in a Gradio chat UI

**How to use this notebook:** Run cells from top to bottom in order (Runtime → Run all,
or run cell-by-cell). GPU is recommended but not required — go to
`Runtime → Change runtime type → T4 GPU` for faster generation.


## 1. Installation

Install all required libraries. This only needs to run once per Colab session.

In [ ]:
# LangChain (text splitting), ChromaDB (vector store), Sentence-Transformers (embeddings),
# Transformers (LLM), pdfplumber (PDF text extraction), Gradio (UI)
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    sentence-transformers \
    transformers \
    accelerate \
    pdfplumber \
    gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 30.2 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import os
import re
import textwrap
from typing import List, Dict

from google.colab import files
import pdfplumber

# Recent langchain releases moved text splitters into their own package;
# try the modern location first and fall back for older installs.
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

import chromadb
from chromadb.config import Settings

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

import gradio as gr

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully.")


All libraries imported successfully.


## 3. PDF Upload & Text Extraction

- `upload_pdf()` opens Colab's file picker and validates that a `.pdf` was chosen.
- `extract_text_from_pdf()` reads every page with `pdfplumber` and raises clear errors for
  missing files, corrupted PDFs, or PDFs with no extractable text (e.g. scanned images
  without OCR). `pdfplumber` is used instead of `pypdf` because some PDFs (custom font
  encodings) cause `pypdf` to mis-decode characters like single-quotes inside code blocks.


In [ ]:
def upload_pdf() -> str:
    """
    Opens a file upload dialog in Colab, saves the uploaded PDF to disk,
    and returns the local file path.
    """
    print("Please select a PDF file to upload...")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file was uploaded. Please try again.")

    pdf_filename = list(uploaded.keys())[0]

    if not pdf_filename.lower().endswith(".pdf"):
        raise ValueError(f"'{pdf_filename}' is not a PDF file. Please upload a .pdf file.")

    print(f"Uploaded: {pdf_filename} ({len(uploaded[pdf_filename]) / 1024:.1f} KB)")
    return pdf_filename


In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extracts raw text from every page of a PDF using pdfplumber.
    pdfplumber is used instead of pypdf because pypdf can mis-decode custom
    font encodings on some PDFs (e.g. turning every single-quote character
    inside code blocks into a literal \"/quotesingle.ts1\" glyph-name string),
    which badly corrupts embeddings. pdfplumber avoids this on the PDFs we've
    tested.
    Raises an error if the file is missing, corrupted, or has no extractable text.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

    try:
        pdf = pdfplumber.open(pdf_path)
    except Exception as e:
        raise ValueError(f"Could not read PDF file. It may be corrupted or encrypted. Error: {e}")

    with pdf:
        if len(pdf.pages) == 0:
            raise ValueError("The PDF has no pages.")

        full_text = []
        for page_num, page in enumerate(pdf.pages):
            try:
                page_text = page.extract_text() or ""
            except Exception as e:
                print(f"Warning: could not extract text from page {page_num + 1}: {e}")
                page_text = ""
            full_text.append(page_text)

        num_pages = len(pdf.pages)

    combined_text = "\n".join(full_text)

    if not combined_text.strip():
        raise ValueError(
            "No extractable text was found in this PDF. "
            "It may be a scanned/image-only document that requires OCR."
        )

    print(f"Extracted text from {num_pages} pages "
          f"({len(combined_text)} characters).")
    return combined_text


## 4. Text Cleaning

Raw PDF extraction is messy: hyphenated line breaks, inconsistent whitespace, and
stray control characters. `clean_text()` normalizes all of that before chunking.

In [ ]:
def clean_text(raw_text: str) -> str:
    """
    Cleans extracted PDF text:
    - fixes hyphenated line breaks (e.g. "exam-\nple" -> "example")
    - collapses line-wrap newlines into spaces (keeps paragraph breaks)
    - collapses runs of 3+ newlines into a single paragraph break
    - strips non-printable control characters
    - collapses repeated spaces/tabs
    - trims whitespace on every line
    """
    text = raw_text

    # Normalize PDF ligature characters (common pypdf extraction artifact)
    # e.g. the single glyph \"ﬁ\" must become \"fi\" or words like \"define\"
    # come out as \"deﬁne\" and break tokenization/embeddings.
    ligature_map = {
        "\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl",
        "\ufb03": "ffi", "\ufb04": "ffl", "\ufb05": "st", "\ufb06": "st",
    }
    for lig, normal in ligature_map.items():
        text = text.replace(lig, normal)

    # Fix words split across a line break by a hyphen
    text = re.sub(r"-\n(?=[a-z])", "", text)

    # Replace single newlines that are just line-wrapping with spaces
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Collapse 3+ newlines into a max of 2 (paragraph break)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove non-printable / control characters
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)

    # Remove table-of-contents \"dot leader\" patterns like \". . . . . . 105\"
    text = re.sub(r"(\.\s*){4,}\d*", " ", text)

    # Collapse multiple spaces/tabs into one
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Strip leading/trailing whitespace on each line
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()


## 5. Chunking with `RecursiveCharacterTextSplitter`

**Parameter choices, explained:**

- **`chunk_size=800`** (characters): a good middle ground. `all-MiniLM-L6-v2` truncates
  input at 256 tokens (~1000-1200 characters), so 800 characters comfortably fits inside
  one embedding pass while still giving the LLM enough surrounding context to answer from.
- **`chunk_overlap=120`** (~15% of chunk size): consecutive chunks share this many
  characters so an idea or sentence that falls right on a chunk boundary isn't cut in half
  and lost to retrieval.
- **`separators=["\n\n", "\n", ". ", " ", ""]`**: the splitter tries each separator in
  order — paragraph breaks first, then line breaks, then sentence boundaries, then words,
  then raw characters as a last resort — so chunks stay as semantically coherent as
  possible instead of splitting mid-sentence.

In [ ]:
def chunk_text(text: str, chunk_size: int = 800, chunk_overlap: int = 120) -> List[str]:
    """
    Splits cleaned text into overlapping chunks using LangChain's
    RecursiveCharacterTextSplitter. See the markdown cell above for why
    these parameter values were chosen.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_text(text)

    # Filter out any tiny/empty fragments that add no retrieval value
    chunks = [c.strip() for c in chunks if len(c.strip()) > 20]

    if not chunks:
        raise ValueError("Text splitting produced no usable chunks.")

    print(f"Split text into {len(chunks)} chunks "
          f"(chunk_size={chunk_size}, chunk_overlap={chunk_overlap}).")
    return chunks


## 6. Embeddings — Sentence-Transformers

Using **`all-MiniLM-L6-v2`**: a lightweight (~80MB), fast, and strong general-purpose
sentence embedding model (384-dimensional vectors) — a great default for Colab-scale
RAG projects.

In [ ]:
print("Loading embedding model: all-MiniLM-L6-v2 ...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded. Embedding dimension:",
      embedding_model.get_sentence_embedding_dimension())

def generate_embeddings(chunks: List[str]) -> List[List[float]]:
    """
    Encodes a list of text chunks into dense vector embeddings.
    Returns a list of plain Python lists (ChromaDB expects lists, not numpy arrays).
    """
    if not chunks:
        raise ValueError("No chunks provided for embedding.")

    embeddings = embedding_model.encode(
        chunks,
        show_progress_bar=True,
        batch_size=32,
        convert_to_numpy=True,
    )
    return embeddings.tolist()


Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded. Embedding dimension: 384


## 7. Vector Database — ChromaDB

Stores, for every chunk: the **document text**, its **embedding**, and **metadata**
(source filename + chunk index). `similarity_search()` embeds the user's question with
the same model and retrieves the closest chunks by cosine similarity.

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
COLLECTION_NAME = "pdf_rag_collection"

def build_vector_store(chunks: List[str], embeddings: List[List[float]],
                        source_filename: str):
    """
    Creates (or resets) a ChromaDB collection and stores document chunks,
    their embeddings, and metadata for each chunk.
    """
    # Drop any previous collection so re-running with a new PDF starts fresh
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    ids = [f"chunk_{i}" for i in range(len(chunks))]
    metadatas = [
        {"source": source_filename, "chunk_index": i}
        for i in range(len(chunks))
    ]

    collection.add(
        ids=ids,
        documents=chunks,
        embeddings=embeddings,
        metadatas=metadatas,
    )

    print(f"Stored {collection.count()} chunks in ChromaDB collection '{COLLECTION_NAME}'.")
    return collection


In [ ]:
def similarity_search(collection, query: str, top_k: int = 4) -> List[Dict]:
    """
    Embeds the query with the same embedding model used for the documents,
    then retrieves the top_k most similar chunks from ChromaDB.
    Returns a list of dicts: {"text": ..., "metadata": ..., "distance": ...}
    """
    if collection is None:
        raise ValueError("No vector store available. Process a PDF first.")

    query_embedding = embedding_model.encode([query], convert_to_numpy=True).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({"text": doc, "metadata": meta, "distance": dist})

    return retrieved


## 8. Language Model — FLAN-T5

`google/flan-t5-base` is a free, instruction-tuned, Colab-friendly model well suited to
extractive/grounded question answering. (Swap in `google/flan-t5-large` if you have a
GPU and want higher quality, at the cost of speed.)

In [ ]:
import torch

LLM_MODEL_NAME = "google/flan-t5-base"

print(f"Loading language model: {LLM_MODEL_NAME} ...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_tokenizer.truncation_side = "left"   # NEW: if truncation is needed, cut from the
                                          # START of the context, never from the end
                                          # where the question lives
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)
print(f"Language model loaded on {device}.")

def generate_answer(prompt: str, max_new_tokens: int = 256) -> str:
    if not prompt or not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    inputs = llm_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=1536  # NEW: raised from 1024
    ).to(device)

    output_ids = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        do_sample=False,
        repetition_penalty=1.3,
        early_stopping=True,
    )

    answer = llm_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer.strip()

Loading language model: google/flan-t5-base ...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language model loaded on cuda.


## 9. Prompt Engineering

The prompt explicitly instructs the model to:
- use **only** the retrieved context (no outside knowledge)
- avoid hallucinating facts, names, or numbers
- reply with a fixed "I don't know" phrase when the answer isn't in the context

In [ ]:
def build_prompt(question: str, retrieved_chunks: List[Dict], max_chars_per_chunk: int = 500) -> str:
    context_text = "\n\n".join(
        f"[Excerpt {i+1}]\n{chunk['text'][:max_chars_per_chunk]}"
        for i, chunk in enumerate(retrieved_chunks)
    )

    prompt = f"""Answer the question using only the facts stated in the Context below.
Write your answer as a normal sentence describing the document's content.
Never repeat these instructions in your answer.
If the Context does not contain the answer, reply only with: I don't know based on the provided document.

Context:
{context_text}

Question: {question}

Answer:"""
    return prompt


## 10. Full RAG Pipeline

Ties every stage together:

```
User Question -> Question Embedding -> ChromaDB Similarity Search ->
Relevant Context Retrieval -> Prompt Construction -> LLM Answer Generation
```

In [ ]:
def rag_pipeline(collection, question: str, top_k: int = 6, verbose: bool = False,
                  distance_threshold: float = 0.65) -> Dict:
    """
    Runs the complete RAG flow and returns the answer plus the retrieved
    context (useful for debugging retrieval quality).

    distance_threshold: if even the closest retrieved chunk is farther than
    this (cosine distance, lower = more similar), we skip the LLM entirely
    and answer \"I don't know\". Small instruction-following models like
    FLAN-T5-base don't reliably say \"I don't know\" on their own when given
    weak/irrelevant context -- they tend to guess something from whatever
    text is in front of them. This threshold catches that case before
    generation. Tune it per document/embedding model: check the printed
    distances in verbose mode and set the threshold just above where
    genuinely relevant matches land.
    """
    if not question or not question.strip():
        return {"answer": "Please enter a question.", "retrieved": [], "prompt": ""}

    retrieved = similarity_search(collection, question, top_k=top_k)

    if not retrieved or retrieved[0]["distance"] > distance_threshold:
        if verbose:
            best = retrieved[0]["distance"] if retrieved else float("inf")
            print(f"\n--- No sufficiently relevant chunks found for: '{question}' "
                  f"(best distance={best:.4f} > threshold={distance_threshold}) ---")
        return {
            "answer": "I don't know based on the provided document.",
            "retrieved": retrieved,
            "prompt": "",
        }

    if verbose:
        print(f"\n--- Retrieved {len(retrieved)} chunks for: '{question}' ---")
        for i, r in enumerate(retrieved):
            print(f"\n[{i+1}] (distance={r['distance']:.4f}, "
                  f"chunk_index={r['metadata']['chunk_index']})")
            print(textwrap.shorten(r["text"], width=200, placeholder="..."))

    prompt = build_prompt(question, retrieved)
    answer = generate_answer(prompt)

    if verbose:
        print(f"\n--- Final Answer ---\n{answer}\n")

    return {"answer": answer, "retrieved": retrieved, "prompt": prompt}


## 11. Run the Ingestion Pipeline

This cell uploads your PDF and runs it through: extract → clean → chunk → embed → store.
Run this once per PDF (re-run it if you want to switch to a different document).

In [ ]:
pdf_path = upload_pdf()
raw_text = extract_text_from_pdf(pdf_path)
cleaned = clean_text(raw_text)
chunks = chunk_text(cleaned, chunk_size=800, chunk_overlap=120)
embeddings = generate_embeddings(chunks)
vector_collection = build_vector_store(chunks, embeddings, source_filename=pdf_path)

print("\n✅ PDF processed and indexed. Ready for questions.")


Please select a PDF file to upload...


Saving pythonlearn.pdf to pythonlearn (5).pdf
Uploaded: pythonlearn (5).pdf (2313.6 KB)
Extracted text from 241 pages (392399 characters).
Split text into 605 chunks (chunk_size=800, chunk_overlap=120).


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Stored 605 chunks in ChromaDB collection 'pdf_rag_collection'.

✅ PDF processed and indexed. Ready for questions.


## 12. Testing & Debugging Retrieval Quality

Runs a few example questions with `verbose=True` so you can see exactly which chunks
were retrieved (and their similarity distance) alongside the generated answer. Edit
`example_questions` to match your own PDF's content.

In [ ]:
example_questions = [
    "What is this document about?",
    "Summarize the main topic in one paragraph.",
    "What is something this document definitely does not mention?",  # sanity check for "I don't know"
]

for q in example_questions:
    result = rag_pipeline(vector_collection, q, top_k=6, verbose=True)
    print("=" * 80)



--- Retrieved 6 chunks for: 'What is this document about?' ---

[1] (distance=0.5058, chunk_index=7)
. This follows a general shift in open documentation licenses moving from the GFDL to the CC-BY-SA (e.g., Wikipedia). Using the CC-BY-SA license maintains the book’s strong copyleft tradition while...

[2] (distance=0.6015, chunk_index=445)
The rest of this chapter will define all of the above terms so make sure to come back after you finish the chapter and re-read the above paragraphs to check your understanding

[3] (distance=0.6133, chunk_index=587)
. Unfortunately, addingNClimitsusesofthismaterialthatIwouldliketopermit. SoIhaveadded this section of the document to describe specific situations where I am giving my permission in advance to use...

[4] (distance=0.6397, chunk_index=578)
. I tried to minimize the jargon and define each term at first use. • Build gradually. To avoid trap doors, I took the most difficult topics and split them into a series of small steps. • Focus on...



In [ ]:
specific_questions = [
    "What is Python according to this book?",
    "What is a variable in Python?",
    "What is the difference between a list and a dictionary?",
    "What does the for loop do in Python?",
    "How do you define a function in Python?",
    "What is a string in Python?",
]

for q in specific_questions:
    result = rag_pipeline(vector_collection, q, top_k=6, verbose=True)
    print("=" * 80)


--- Retrieved 6 chunks for: 'What is Python according to this book?' ---

[1] (distance=0.3372, chunk_index=6)
. 1Except,ofcourse,forthisline. iv Students who find this book interesting and want to further explore should look at Allen B. Downey’s Think Python book. Because there is a lot of overlap between...

[2] (distance=0.3515, chunk_index=581)
. A.4.2 Acknowledgements for “Think Python” (Allen B. Downey) Firstandmostimportantly,IthankJeffElkner,whotranslatedmyJavabookinto Python, which got this project started and introduced me to what...

[3] (distance=0.3936, chunk_index=48)
. Now at this point in our discussion of compilers and interpreters, you should be wondering a bit about the Python interpreter itself. What language is it written in? Is it written in a compiled...

[4] (distance=0.4082, chunk_index=441)
. The key outcome of this chapter is to have a basic understanding of how objects are constructed and how they function and most importantly how we make use of the capabil

## 13. Gradio Chatbot Interface

A simple chat UI with:
- a PDF upload + "Process PDF" button
- a chat window with full conversation history
- a question box
- loading messages while the PDF is processed and while an answer is generated
- retrieved-context preview appended to each answer, for transparency/debugging

You can re-upload a different PDF at any time and click **Process PDF** again to
re-index; the chatbot will then answer from the new document.

In [ ]:
# Global app state (fine for a single-user Colab session)
app_state = {"collection": None, "filename": None}


def process_pdf_ui(pdf_file):
    if pdf_file is None:
        yield "⚠️ Please upload a PDF file first.", gr.update(interactive=False)
        return

    yield ("⏳ Processing PDF... extracting text, chunking, and building "
           "embeddings. This may take a minute."), gr.update(interactive=False)

    try:
        pdf_path = pdf_file.name if hasattr(pdf_file, "name") else pdf_file
        raw = extract_text_from_pdf(pdf_path)
        cleaned_local = clean_text(raw)
        local_chunks = chunk_text(cleaned_local)
        local_embeddings = generate_embeddings(local_chunks)
        collection = build_vector_store(
            local_chunks, local_embeddings, source_filename=os.path.basename(pdf_path)
        )

        app_state["collection"] = collection
        app_state["filename"] = os.path.basename(pdf_path)

        yield (f"✅ Processed '{app_state['filename']}' into {len(local_chunks)} chunks. "
               f"You can now ask questions below."), gr.update(interactive=True)
    except Exception as e:
        yield f"❌ Error processing PDF: {e}", gr.update(interactive=False)


def chat_respond(message, history):
    """
    Uses the dict-based messages format ({"role": ..., "content": ...}),
    which this Gradio version expects internally even though its
    Chatbot() constructor doesn't accept a `type=` argument.
    """
    if history is None:
        history = []

    if not message or not message.strip():
        yield history, ""
        return

    if app_state["collection"] is None:
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "⚠️ Please upload and process a PDF first."},
        ]
        yield history, ""
        return

    interim = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": "⏳ Thinking..."},
    ]
    yield interim, ""

    try:
        result = rag_pipeline(app_state["collection"], message, top_k=6)
        answer = result["answer"]

        sources_preview = "\n\n---\n**Retrieved context (top matches):**\n"
        for i, r in enumerate(result["retrieved"]):
            snippet = textwrap.shorten(r["text"], width=150, placeholder="...")
            sources_preview += f"\n{i+1}. (distance={r['distance']:.3f}) {snippet}"

        full_answer = f"{answer}{sources_preview}"
    except Exception as e:
        full_answer = f"❌ Error generating answer: {e}"

    final = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": full_answer},
    ]
    yield final, ""


with gr.Blocks(title="PDF RAG Chatbot") as demo:
    gr.Markdown("# 📚 PDF RAG Chatbot\nUpload a PDF, click **Process PDF**, then ask questions grounded in its content.")

    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        process_btn = gr.Button("Process PDF", variant="primary")

    status_box = gr.Markdown("No PDF processed yet.")

    chatbot = gr.Chatbot(label="Chat History", height=450)  # no `type=` argument
    question_box = gr.Textbox(
        label="Ask a question",
        placeholder="e.g. What is the main topic of chapter 2?",
        interactive=False,
    )

    with gr.Row():
        ask_btn = gr.Button("Ask")
        clear_btn = gr.Button("Clear Chat")

    process_btn.click(fn=process_pdf_ui, inputs=[pdf_input], outputs=[status_box, question_box])
    ask_btn.click(fn=chat_respond, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    question_box.submit(fn=chat_respond, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    clear_btn.click(lambda: [], None, chatbot)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cf9b27a9a823014b1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Extracted text from 241 pages (392399 characters).
Split text into 605 chunks (chunk_size=800, chunk_overlap=120).


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Stored 605 chunks in ChromaDB collection 'pdf_rag_collection'.
Extracted text from 241 pages (392399 characters).
Split text into 605 chunks (chunk_size=800, chunk_overlap=120).


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Stored 605 chunks in ChromaDB collection 'pdf_rag_collection'.


## Notes & Next Steps

- **Swap models:** for higher-quality answers, try `google/flan-t5-large` or
  `google/flan-t5-xl` (needs more GPU memory), or point `generate_answer` at an API-based
  model.
- **Persist the vector store:** replace `chromadb.Client(...)` with
  `chromadb.PersistentClient(path="/content/chroma_db")` to keep the index across
  Colab sessions.
- **Multiple PDFs:** loop `extract_text_from_pdf` + `chunk_text` over several files and
  add a `"source"` metadata field per file before calling `build_vector_store`, so answers
  can cite which document they came from.
- **Scanned PDFs:** if `extract_text_from_pdf` raises a "no extractable text" error, run
  OCR (e.g. `pytesseract`) on the PDF pages first.
